# 0.0 Initial Library Import


In [ ]:
# Standard libraries
import sys, math, itertools, warnings, importlib, textwrap, random, ast, re, gc, pickle, json, os, sklearn, xgboost, joblib
import ast
import importlib
import itertools
import json
import math
import os
import pickle
import random
import re
import sys
import textwrap
import warnings
from datetime import date, datetime
from pathlib import Path
from typing import Any, Dict, List

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
NOTEBOOK_UTILS_SRC = PROJECT_ROOT / "notebook_utils" / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

if str(NOTEBOOK_UTILS_SRC) not in sys.path:
    sys.path.append(str(NOTEBOOK_UTILS_SRC))

from paths import DATA_INTERMEDIATE, DATA_RAW, FIGURES, MODELS, PROJECT_ROOT, TABLES

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyfolio as pf
import seaborn as sns
from IPython.display import HTML, display
from scipy.optimize import minimize
from scipy.stats import entropy, norm, t

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [ ]:
# Panda display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)
pd.options.display.float_format = "{:,.5f}".format
np.set_printoptions(precision=5, suppress=True)

print("pandas:", pd.__version__)
print("numpy :", np.__version__)


# 0.1 Research Config, Paths, and Guardrails


In [ ]:
# Model setup, repo-relative paths, notebook constants, and light notebook guardrails
model = "regime_model"
model_round = "round_1"
RUN_LABEL = "static_opt"

MAX_CAPITAL = 68500
ENTRY_FEE = 3.5
LONG_SHORT = -1
MIN_TRADES = 25

GREEDY_N_QUANTILES = 24
GREEDY_MAX_FEATURES = 6
GREEDY_IMPROVEMENT_EPS = 0.02
SEARCH_MIN_COUNT_PER_SIDE = 200
ROUND_CRIT_DIGITS = 4

BAYESIAN_N_CALLS_PER_STEP = 35
BAYESIAN_N_RANDOM_STARTS = 10
BAYESIAN_TAIL_PENALTY = 0.4
BAYESIAN_COMPLEXITY_PENALTY = 0.03
BAYESIAN_LOCK_TO_GRID = True
BAYESIAN_N_QUANTILES_LOCK = 32

save_best_json = False
save_equity_curve = False
save_partition_dashboards_debug = False

model_input_path = DATA_RAW / "xls" / "input" / model
model_round_input_path = model_input_path / model_round
model_intermediate_path = DATA_INTERMEDIATE / model / model_round
model_table_output_path = TABLES / model / model_round
model_csv_output_path = model_table_output_path / "csv"
model_xls_output_path = model_table_output_path / "xls"
model_figure_output_path = FIGURES / model / model_round
model_ml_output_path = MODELS / model / model_round
research_liquidity_path = DATA_RAW / "research" / "liquidity" / "xls"
research_fear_greed_path = DATA_RAW / "research" / "fear_greed" / "csv"

pass3_output_path = model_table_output_path / RUN_LABEL
pass3_csv_output_path = pass3_output_path / "csv"
pass3_json_output_path = pass3_output_path / "json"

for directory in [
    model_intermediate_path,
    model_table_output_path,
    model_csv_output_path,
    model_xls_output_path,
    model_figure_output_path,
    model_ml_output_path,
    pass3_output_path,
    pass3_csv_output_path,
    pass3_json_output_path,
]:
    directory.mkdir(parents=True, exist_ok=True)

PARTITION_NAMES = [
    "partition_ins_80_001",
    "partition_ins_20_001",
    "partition_oos_001",
]

PARTITION_OUTPUT_LABELS = {
    "partition_ins_80_001": "ins_80_1",
    "partition_ins_20_001": "ins_20_1",
    "partition_oos_001": "oos_1",
}

REQUIRED_BASE_COLUMNS = [
    "normed_date",
    "symbol",
    "entry_time",
    "entry_price",
    "entry_shares",
    "exit_price",
    "exit_shares",
    "matched_shares",
    "mtm_pl",
    "entry_pl",
    "entry_side",
    "entry_fees",
    "exit_fees",
    "pl_g",
    "pl_n",
    "fees",
]

def assert_file_exists(path: Path, label: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    return path

def ensure_columns(df: pd.DataFrame, required_columns, df_name: str) -> None:
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise KeyError(f"{df_name} is missing required columns: {missing_columns}")

def ensure_non_empty(df: pd.DataFrame, df_name: str) -> None:
    if df.empty:
        raise ValueError(f"{df_name} is empty. Check the upstream parquet export.")

def ensure_partition_ready(df: pd.DataFrame, df_name: str) -> None:
    ensure_non_empty(df, df_name)
    ensure_columns(df, REQUIRED_BASE_COLUMNS, df_name)

def safe_feature_subset(available_columns, candidate_columns):
    available = set(available_columns)
    return [col for col in candidate_columns if col in available]


# 0.2 Project Utilities


In [ ]:
FEATURE_THRESHOLD_OVERRIDES = {}

def normalize_crit_list(crit_list):
    if crit_list is None:
        return []
    if isinstance(crit_list, str):
        crit_list = ast.literal_eval(crit_list)
    if isinstance(crit_list, dict):
        crit_list = [crit_list]

    normalized = []
    for item in crit_list:
        if not isinstance(item, dict):
            continue
        normalized_item = {}
        for key, value in item.items():
            if isinstance(value, (list, tuple)) and len(value) == 2:
                threshold, operator_name = value
                normalized_item[str(key)] = (float(threshold), str(operator_name))
        if normalized_item:
            normalized.append(normalized_item)
    return normalized

def crit_list_to_rule_dict(crit_list):
    crit_list = normalize_crit_list(crit_list)
    combined = {}
    for item in crit_list:
        combined.update(item)
    return combined

def rule_dict_to_crit_list(rule_dict):
    return [dict(rule_dict)] if rule_dict else []

def crit_list_to_literal(crit_list):
    return repr(crit_list_to_rule_dict(crit_list))

def chosen_to_crit_list(chosen):
    rule_dict = {
        candidate.feat: (float(candidate.threshold), str(candidate.op))
        for candidate in chosen
    }
    return rule_dict_to_crit_list(rule_dict)

def json_ready(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, np.ndarray):
        return [json_ready(item) for item in value.tolist()]
    if isinstance(value, dict):
        return {str(key): json_ready(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value

def save_dataframe_csv(df: pd.DataFrame, filename: str, output_dir: Path = pass3_csv_output_path) -> Path:
    output_path = output_dir / filename
    df.to_csv(output_path, index=False)
    return output_path

def save_json(payload: Dict[str, Any], filename: str, output_dir: Path = pass3_json_output_path) -> Path:
    output_path = output_dir / filename
    with output_path.open("w", encoding="utf-8") as handle:
        json.dump(json_ready(payload), handle, indent=2)
    return output_path

def extract_dashboard_metrics(strat_df: pd.DataFrame) -> Dict[str, Any]:
    if strat_df is None or strat_df.empty:
        return {}
    metrics = strat_df.copy()
    metrics["metric"] = metrics["metric"].astype(str)
    return dict(zip(metrics["metric"], metrics["value"]))

def is_monotonic_equity_curve(bydate_df: pd.DataFrame, curve_col: str = "cum_pl_n") -> bool:
    if bydate_df is None or bydate_df.empty or curve_col not in bydate_df.columns:
        return False
    curve = bydate_df[curve_col].dropna()
    if curve.empty:
        return False
    increments = curve.diff().fillna(curve.iloc[0])
    return bool((increments >= 0).all())

def round_threshold_for_feature(feature_name: str, value: float, overrides: Dict[str, Any] = None):
    overrides = overrides or FEATURE_THRESHOLD_OVERRIDES
    if feature_name in overrides:
        override = overrides[feature_name]
        return override(value) if callable(override) else round(float(value), int(override))

    feature_lower = feature_name.lower()
    value = float(value)
    abs_value = abs(value)

    if "proba" in feature_lower or "probability" in feature_lower:
        return round(value, 2)

    if feature_lower in {"fear_greed", "rsi"} or feature_lower.endswith("_rsi"):
        return int(round(value))

    if any(token in feature_lower for token in ["vol", "rvol", "atr", "natr", "beta", "cmf", "kalmar", "scaled"]):
        if abs_value < 10:
            return round(value, 2)
        if abs_value < 100:
            return round(value, 1)
        return int(round(value))

    if feature_lower.startswith("ret_") or feature_lower.startswith("pct_") or feature_lower.startswith("dist_"):
        if abs_value < 1:
            return round(value, 3)
        if abs_value < 10:
            return round(value, 2)
        return round(value, 1)

    if abs_value < 1:
        return round(value, 3)
    if abs_value < 10:
        return round(value, 2)
    if abs_value < 100:
        return round(value, 1)
    return int(round(value))

def round_crit_list(crit_list, overrides: Dict[str, Any] = None):
    rule_dict = crit_list_to_rule_dict(crit_list)
    rounded_rule_dict = {
        feature_name: (round_threshold_for_feature(feature_name, threshold, overrides=overrides), operator_name)
        for feature_name, (threshold, operator_name) in rule_dict.items()
    }
    return rule_dict_to_crit_list(rounded_rule_dict)

def evaluate_rule_metrics(df: pd.DataFrame, crit_list):
    return _fast_optimization_v1.evaluate_threshold_rule(
        df,
        normalize_crit_list(crit_list),
        mtm_col="mtm_pl",
        date_col="normed_date",
        use_numexpr=True,
        crit_round_digits=ROUND_CRIT_DIGITS,
    )

def summarize_partition_rule(
    df: pd.DataFrame,
    crit_list,
    partition_name: str,
    method_name: str,
    candidate_name: str,
):
    crit_list = normalize_crit_list(crit_list)
    rule_dict = crit_list_to_rule_dict(crit_list)
    missing_rule_columns = [col for col in rule_dict if col not in df.columns]
    if missing_rule_columns:
        raise KeyError(f"{partition_name} is missing rule columns: {missing_rule_columns}")

    shared_metrics = evaluate_rule_metrics(df, crit_list)
    trade_count = int(shared_metrics["trade_count"])
    sharpe = float(shared_metrics["sharpe"]) if np.isfinite(shared_metrics["sharpe"]) else np.nan
    ann_return = float(shared_metrics["ann_return"]) if np.isfinite(shared_metrics["ann_return"]) else np.nan
    ann_vol = float(shared_metrics["ann_vol"]) if np.isfinite(shared_metrics["ann_vol"]) else np.nan
    max_dd = float(shared_metrics["max_dd"]) if np.isfinite(shared_metrics["max_dd"]) else np.nan
    capital_used = float(shared_metrics["capital_used"])

    optmz_summary, optmz_events, optmz_bydate = optmz_loop_wrap(df, crit_list, MAX_CAPITAL)
    if isinstance(optmz_bydate, tuple):
        optmz_bydate = None

    if optmz_events is None or isinstance(optmz_events, tuple) or len(optmz_events) == 0:
        return {
            "summary_row": {
                "method_name": method_name,
                "candidate_name": candidate_name,
                "partition_name": partition_name,
                "partition_label": PARTITION_OUTPUT_LABELS[partition_name],
                "trade_count": trade_count,
                "sharpe": sharpe,
                "ann_return": ann_return,
                "ann_vol": ann_vol,
                "max_dd": max_dd,
                "capital_used": capital_used,
                "dashboard_ret_n": np.nan,
                "dashboard_annual_ret": np.nan,
                "dashboard_win_rate": np.nan,
                "dashboard_drawdown_dollar": np.nan,
                "dashboard_pnl_n": np.nan,
                "monotonicity_flag": False,
                "passes_min_trades": trade_count >= MIN_TRADES,
                "passes_positive_sharpe": bool(pd.notna(sharpe) and sharpe > 0),
                "passes_monotonicity": False,
                "validation_pass": False,
                "crit_list_json": crit_list_to_literal(crit_list),
            },
            "dashboard_df": pd.DataFrame(),
            "dashboard_bydate_df": pd.DataFrame(),
            "optmz_summary_df": optmz_summary if isinstance(optmz_summary, pd.DataFrame) else pd.DataFrame(),
            "rule_metrics": shared_metrics,
        }

    strat_df, dashboard_event_df, dashboard_bydate_df = dashboard(
        optmz_events,
        ENTRY_FEE,
        f"{method_name}_{candidate_name}_{PARTITION_OUTPUT_LABELS[partition_name]}",
        LONG_SHORT,
        "complete",
        "max",
    )

    dashboard_metrics = extract_dashboard_metrics(strat_df)
    monotonicity_flag = is_monotonic_equity_curve(dashboard_bydate_df, "cum_pl_n")

    return {
        "summary_row": {
            "method_name": method_name,
            "candidate_name": candidate_name,
            "partition_name": partition_name,
            "partition_label": PARTITION_OUTPUT_LABELS[partition_name],
            "trade_count": trade_count,
            "sharpe": sharpe,
            "ann_return": ann_return,
            "ann_vol": ann_vol,
            "max_dd": max_dd,
            "capital_used": capital_used,
            "dashboard_ret_n": dashboard_metrics.get("ret_n", np.nan),
            "dashboard_annual_ret": dashboard_metrics.get("annual ret", np.nan),
            "dashboard_win_rate": dashboard_metrics.get("daily win", np.nan),
            "dashboard_drawdown_dollar": dashboard_metrics.get("max drawday dollar", np.nan),
            "dashboard_pnl_n": dashboard_metrics.get("pl_n", np.nan),
            "monotonicity_flag": monotonicity_flag,
            "passes_min_trades": bool(trade_count >= MIN_TRADES),
            "passes_positive_sharpe": bool(pd.notna(sharpe) and sharpe > 0),
            "passes_monotonicity": monotonicity_flag,
            "validation_pass": bool(
                trade_count >= MIN_TRADES and pd.notna(sharpe) and sharpe > 0 and monotonicity_flag
            ),
            "crit_list_json": crit_list_to_literal(crit_list),
        },
        "dashboard_df": strat_df,
        "dashboard_bydate_df": dashboard_bydate_df,
        "optmz_summary_df": optmz_summary,
        "rule_metrics": shared_metrics,
    }


In [ ]:
def build_search_record(
    method_name: str,
    candidate_name: str,
    feature_pool: List[str],
    search_result,
    raw_validation: Dict[str, Any],
    rounded_validation: Dict[str, Any],
) -> Dict[str, Any]:
    rounded_summary = rounded_validation["summary_row"]
    raw_summary = raw_validation["summary_row"]
    return {
        "method_name": method_name,
        "candidate_name": candidate_name,
        "feature_pool_count": len(feature_pool),
        "selected_feature_count": len(search_result.chosen),
        "crit_list_json": rounded_summary["crit_list_json"],
        "search_trade_count": search_result.trade_count,
        "search_sharpe": search_result.sharpe,
        "search_ann_return": search_result.ann_return,
        "search_ann_vol": search_result.ann_vol,
        "search_max_dd": search_result.max_dd,
        "search_capital_used": search_result.capital_used,
        "reapplied_trade_count_ins_80": raw_summary["trade_count"],
        "reapplied_sharpe_ins_80": raw_summary["sharpe"],
        "reapplied_capital_used_ins_80": raw_summary["capital_used"],
        "search_apply_trade_delta_ins_80": raw_summary["trade_count"] - search_result.trade_count,
        "search_apply_sharpe_delta_ins_80": raw_summary["sharpe"] - search_result.sharpe,
        "rounded_trade_count_ins_80": rounded_summary["trade_count"],
        "rounded_sharpe_ins_80": rounded_summary["sharpe"],
        "rounded_ann_return_ins_80": rounded_summary["ann_return"],
        "rounded_ann_vol_ins_80": rounded_summary["ann_vol"],
        "rounded_max_dd_ins_80": rounded_summary["max_dd"],
        "rounded_capital_used_ins_80": rounded_summary["capital_used"],
        "monotonicity_flag_ins_80": rounded_summary["monotonicity_flag"],
        "passes_min_trades_ins_80": rounded_summary["passes_min_trades"],
        "passes_positive_sharpe_ins_80": rounded_summary["passes_positive_sharpe"],
        "passes_monotonicity_ins_80": rounded_summary["passes_monotonicity"],
        "validation_pass_ins_80": rounded_summary["validation_pass"],
        "max_capital_global": search_result.logs.get("max_capital_global"),
    }

def export_search_artifacts(method_name: str, candidate_df: pd.DataFrame):
    ranked_df = candidate_df.sort_values(
        by=[
            "validation_pass_ins_80",
            "passes_positive_sharpe_ins_80",
            "passes_monotonicity_ins_80",
            "passes_min_trades_ins_80",
            "rounded_sharpe_ins_80",
            "search_sharpe",
        ],
        ascending=[False, False, False, False, False, False],
    ).reset_index(drop=True)

    csv_path = save_dataframe_csv(ranked_df, f"{method_name}_candidates.csv")
    json_path = None
    if save_best_json and not ranked_df.empty:
        json_path = save_json(ranked_df.iloc[0].to_dict(), f"{method_name}_best.json")
    return ranked_df, csv_path, json_path

import notebook_utils._formatting_functions as _formatting_functions
importlib.reload(_formatting_functions)
from notebook_utils._formatting_functions import nan_inf_summary

import notebook_utils._optimization_functions_37_v2 as _optimization_functions_37_v2
importlib.reload(_optimization_functions_37_v2)
from notebook_utils._optimization_functions_37_v2 import (
    optimization_ranges,
    process_optmz_minmax,
    optmz_loop_wrap,
    process_optmz_breaks,
    optmz_search_with_exclusions,
    sum_combinations,
    create_combinations,
    text_to_dict,
    outlier_bound,
    optmz_loop_wrap_with_exclusions,
)

import notebook_utils._fast_optimization_v1 as _fast_optimization_v1
importlib.reload(_fast_optimization_v1)
from notebook_utils._fast_optimization_v1 import greedy_threshold_search

import notebook_utils._bayesian_optimization_v1 as _bayesian_optimization_v1
importlib.reload(_bayesian_optimization_v1)
from notebook_utils._bayesian_optimization_v1 import bayesian_greedy_threshold_search

import notebook_utils._dashboard_functions_one_symbol_v2 as _dashboard_functions_one_symbol_v2
importlib.reload(_dashboard_functions_one_symbol_v2)
from notebook_utils._dashboard_functions_one_symbol_v2 import dashboard

import notebook_utils._data_explore_functions as _data_explore_functions
importlib.reload(_data_explore_functions)
from notebook_utils._data_explore_functions import (
    cross_tabs,
    explore_cross,
    decile_summary,
    line_chart_grid,
    create_distance,
    append_summary,
    get_summary,
    reset_summary,
    count_outliers_by_std,
)


In [ ]:
# Main dictionary for time-serie data aggregation
sum_cols = ["mtm_pl", "entry_pl", "matched_shares", "entry_side", "entry_fees", "exit_fees", "exit_shares", "pl_g", "pl_n", "fees"]
sum_dict = {key: "sum" for key in sum_cols}


# 0.3 Partition Loading and Validation


In [ ]:
partition_paths = {
    name: assert_file_exists(model_intermediate_path / f"{name}.parquet", f"{name} parquet")
    for name in PARTITION_NAMES
}

loaded = {
    name: pd.read_parquet(path, engine="pyarrow")
    for name, path in partition_paths.items()
}

for partition_name, partition_df in loaded.items():
    ensure_partition_ready(partition_df, partition_name)

partition_ins_80_001 = loaded["partition_ins_80_001"]
partition_ins_20_001 = loaded["partition_ins_20_001"]
partition_oos_001 = loaded["partition_oos_001"]

print({key: value.shape for key, value in loaded.items()})


In [ ]:
partition_ins_80_001.head(2)


# 1.0 Feature Universe Construction


In [ ]:
GROUP_ALWAYS = [
    "atr_250",
    "atr_ema_a",
    "atr_ema_b",
    "atr_ktg",
    "atr_sma_a",
    "atr_sma_b",
    "beta",
    "cmf",
    "corr_1y",
    "corr_20d",
    "d_atr",
    "d_avol5",
    "d_avol50",
    "d_natr",
    "d_natr_ktg",
    "d_rsi",
    "kalmar_q",
    "pct_chg_open",
    "rsi",
    "rvol",
    "vol_20d",
    "vol_5d",
    "vol_60d",
    "spy_atr",
    "spy_rvol",
    "PCA_Index_ma5",
    "PCA_ScaledIndex_ma5",
    "PCA_Index_ma20",
    "PCA_ScaledIndex_ma20",
    "PCA_Index_ma50",
    "PCA_ScaledIndex_ma50",
    "PCA_Raw_full",
    "PCA_Index_full",
    "fear_greed",
]

GROUP_CALENDAR = [
    "entry_hr_dec",
    "entry_hr_dec_to_close",
    "week_day_sin",
    "week_day_cos",
    "month_sin",
    "month_cos",
    "year_day_sin",
    "year_day_cos",
]

EXCLUDE_COLUMNS = {
    "mtm_pl",
    "pl_g",
    "pl_n",
    "fees",
    "Capital",
    "wins",
    "target_return",
    "target_down",
    "source_file",
    "entry_time",
    "exit_time",
    "normed_date",
    "symbol",
}

FEATURE_RULES = {
    "distance": lambda c: c.startswith("dist_"),
    "percent": lambda c: c.startswith("pct_"),
    "return": lambda c: c.startswith("ret_"),
    "dummy": lambda c: c.startswith("dumm_"),
    "volume_ratio": lambda c: ("vol" in c and "_rat" in c),
}

MANUAL_INCLUDE = []
MANUAL_EXCLUDE = []

def build_feature_list_from_columns(df):
    cols = df.columns.tolist()
    selected = []
    selected.extend(safe_feature_subset(cols, GROUP_ALWAYS))
    selected.extend(safe_feature_subset(cols, GROUP_CALENDAR))

    for _, rule in FEATURE_RULES.items():
        selected.extend([col for col in cols if rule(col)])

    selected.extend(safe_feature_subset(cols, MANUAL_INCLUDE))
    selected = list(dict.fromkeys(selected))
    selected = [col for col in selected if col not in EXCLUDE_COLUMNS]
    selected = [col for col in selected if col not in MANUAL_EXCLUDE]
    return selected


In [ ]:
# Note: "_001" suffix means insample ("ins") train ("80") data with ML probability (w. ml_proba_1)
ins80_feature_columns = build_feature_list_from_columns(partition_ins_80_001)
common_feature_columns = [
    col
    for col in ins80_feature_columns
    if col in partition_ins_20_001.columns and col in partition_oos_001.columns
]

dropped_feature_columns = [col for col in ins80_feature_columns if col not in common_feature_columns]
feature_columns = common_feature_columns

if not feature_columns:
    raise ValueError("No shared feature columns were selected from the current partitions.")

ensure_columns(partition_ins_80_001, feature_columns, "partition_ins_80_001 feature universe")
ensure_columns(partition_ins_20_001, feature_columns, "partition_ins_20_001 feature universe")
ensure_columns(partition_oos_001, feature_columns, "partition_oos_001 feature universe")

probs = [
    col
    for col in ["ml_proba_1"]
    if col in partition_ins_80_001.columns and col in partition_ins_20_001.columns and col in partition_oos_001.columns
]
optmz_list_all_plus = list(dict.fromkeys(feature_columns + probs))

feats_a = safe_feature_subset(feature_columns, GROUP_ALWAYS)
feats_b = [col for col in feature_columns if col.startswith("dist_")]
feats_c = [col for col in feature_columns if col.startswith("pct_")]
feats_d = [col for col in feature_columns if col.startswith("ret_")]

feats_a_ml = list(dict.fromkeys(feats_a + probs))
feats_b_ml = list(dict.fromkeys(feats_b + probs))
feats_c_ml = list(dict.fromkeys(feats_c + probs))
feats_d_ml = list(dict.fromkeys(feats_d + probs))

feature_set_registry = {
    "all_plus_ml": optmz_list_all_plus,
    "no_ml": feature_columns,
    "branch_a": feats_a,
    "branch_b": feats_b,
    "branch_c": feats_c,
    "branch_d": feats_d,
    "branch_a_ml": feats_a_ml,
    "branch_b_ml": feats_b_ml,
    "branch_c_ml": feats_c_ml,
    "branch_d_ml": feats_d_ml,
}
feature_set_registry = {name: cols for name, cols in feature_set_registry.items() if cols}

print(f"shared variables: {len(feature_columns)}")
print(textwrap.fill(", ".join(feature_columns), width=250))
print("")
print(f"ml probability columns included: {probs}")
print(f"dropped because not shared across partitions: {len(dropped_feature_columns)}")


# 1.1 Optimization Range Generation


In [ ]:
drop_manual = []
all_ranges, var_stats = optimization_ranges(partition_ins_80_001, optmz_list_all_plus, drop_manual, 20)
all_ranges = all_ranges.loc[all_ranges["steps"] != 0].copy()

all_ranges_export_path = model_xls_output_path / f"optmz ranges {datetime.now().strftime('%Y%m%d')}.xlsx"
all_ranges.to_excel(all_ranges_export_path, index=True, engine="openpyxl")

print(f"ranges exported to: {all_ranges_export_path}")
all_ranges.tail(5)


# 1.2 Single-Variable Exploration


In [ ]:
target_vars = {index: (row[0.1], row[0.9], row["steps"]) for index, row in all_ranges.iterrows()}
print(f"Target variables: {len(target_vars)}")

crit_lst = [
    {key: (j, ">=")}
    for key, (element1, element2, element3) in target_vars.items()
    for j in np.arange(element1, element2, element3)
]
print(f"Search criteria: {len(crit_lst)}")

single_pos0, optmz, optmz_bydate = optmz_loop_wrap(partition_ins_80_001, crit_lst, MAX_CAPITAL)
single_pos_extrm, single_pos_all = process_optmz_minmax(single_pos0, "Sharpe ratio", -3)
gc.collect()

print(f"{len(target_vars)} variables x 20 steps: {len(single_pos_all)}")
print(f"{len(target_vars)} variables x 2 extreme values: {len(single_pos_extrm)}")

single_variable_export_path = model_xls_output_path / f"optimization all result - pos {datetime.now().strftime('%Y%m%d')} v1.xlsx"
single_pos_all.to_excel(single_variable_export_path, index=True, engine="openpyxl")
print(f"single-variable exploration exported to: {single_variable_export_path}")

single_pos_all.head(10)


# 1.3 Greedy Threshold Search


In [ ]:
greedy_search_results = {}
greedy_search_records = []

for candidate_name, candidate_features in feature_set_registry.items():
    res = greedy_threshold_search(
        partition_ins_80_001,
        candidate_features,
        n_quantiles=GREEDY_N_QUANTILES,
        max_features=GREEDY_MAX_FEATURES,
        improvement_eps=GREEDY_IMPROVEMENT_EPS,
        adaptive_tails=True,
        min_count_per_side=SEARCH_MIN_COUNT_PER_SIDE,
        crit_round_digits=ROUND_CRIT_DIGITS,
    )
    raw_crit_list = chosen_to_crit_list(res.chosen)
    raw_validation = summarize_partition_rule(
        partition_ins_80_001,
        raw_crit_list,
        "partition_ins_80_001",
        "greedy_threshold_search",
        candidate_name,
    )
    rounded_crit_list = round_crit_list(res.crit_list)
    rounded_validation = summarize_partition_rule(
        partition_ins_80_001,
        rounded_crit_list,
        "partition_ins_80_001",
        "greedy_threshold_search",
        candidate_name,
    )

    greedy_search_results[candidate_name] = {
        "search_result": res,
        "raw_validation": raw_validation,
        "rounded_crit_list": rounded_crit_list,
        "rounded_validation": rounded_validation,
    }
    greedy_search_records.append(
        build_search_record(
            "greedy_threshold_search",
            candidate_name,
            candidate_features,
            res,
            raw_validation,
            rounded_validation,
        )
    )

    print(candidate_name)
    print("Raw chosen:", [(c.feat, c.op, round(c.threshold, 4)) for c in res.chosen])
    print("Rounded rule:", crit_list_to_literal(rounded_crit_list))
    print(
        "Search Sharpe={:.3f}  Reapplied Raw Sharpe={:.3f}  Rounded Sharpe={:.3f}".format(
            res.sharpe,
            raw_validation["summary_row"]["sharpe"],
            rounded_validation["summary_row"]["sharpe"],
        )
    )
    print(
        "Counts: search={} raw_reapply={} rounded={}".format(
            res.trade_count,
            raw_validation["summary_row"]["trade_count"],
            rounded_validation["summary_row"]["trade_count"],
        )
    )
    print(
        "Capital: search={:.2f} raw_reapply={:.2f} rounded={:.2f}".format(
            res.capital_used,
            raw_validation["summary_row"]["capital_used"],
            rounded_validation["summary_row"]["capital_used"],
        )
    )
    print("")

greedy_candidates_df = pd.DataFrame(greedy_search_records)
greedy_candidates_ranked_df, greedy_candidates_csv_path, greedy_best_json_path = export_search_artifacts(
    "greedy_threshold_search",
    greedy_candidates_df,
)

print(f"greedy candidates exported to: {greedy_candidates_csv_path}")
if greedy_best_json_path is not None:
    print(f"greedy best rule exported to: {greedy_best_json_path}")

greedy_candidates_ranked_df.head(2)


# 1.4 Bayesian Greedy Threshold Search


In [ ]:
bayesian_feature_sets = {
    key: value
    for key, value in feature_set_registry.items()
    if key not in {"all_plus_ml", "no_ml"}
}

bayesian_search_results = {}
bayesian_search_records = []

for candidate_name, candidate_features in bayesian_feature_sets.items():
    res = bayesian_greedy_threshold_search(
        df=partition_ins_80_001,
        features=candidate_features,
        mtm_col="mtm_pl",
        date_col="normed_date",
        max_features=GREEDY_MAX_FEATURES,
        improvement_eps=GREEDY_IMPROVEMENT_EPS,
        seed=123,
        use_numexpr=True,
        n_calls_per_step=BAYESIAN_N_CALLS_PER_STEP,
        n_random_starts=BAYESIAN_N_RANDOM_STARTS,
        tail_penalty=BAYESIAN_TAIL_PENALTY,
        complexity_penalty=BAYESIAN_COMPLEXITY_PENALTY,
        lock_to_grid=BAYESIAN_LOCK_TO_GRID,
        n_quantiles_lock=BAYESIAN_N_QUANTILES_LOCK,
        crit_round_digits=ROUND_CRIT_DIGITS,
    )
    raw_crit_list = chosen_to_crit_list(res.chosen)
    raw_validation = summarize_partition_rule(
        partition_ins_80_001,
        raw_crit_list,
        "partition_ins_80_001",
        "bayesian_greedy_threshold_search",
        candidate_name,
    )
    rounded_crit_list = round_crit_list(res.crit_list)
    rounded_validation = summarize_partition_rule(
        partition_ins_80_001,
        rounded_crit_list,
        "partition_ins_80_001",
        "bayesian_greedy_threshold_search",
        candidate_name,
    )

    bayesian_search_results[candidate_name] = {
        "search_result": res,
        "raw_validation": raw_validation,
        "rounded_crit_list": rounded_crit_list,
        "rounded_validation": rounded_validation,
    }
    bayesian_search_records.append(
        build_search_record(
            "bayesian_greedy_threshold_search",
            candidate_name,
            candidate_features,
            res,
            raw_validation,
            rounded_validation,
        )
    )

    print(candidate_name)
    print("Raw chosen:", [(c.feat, c.op, round(c.threshold, 4)) for c in res.chosen])
    print("Rounded rule:", crit_list_to_literal(rounded_crit_list))
    print(
        "Search Sharpe={:.3f}  Reapplied Raw Sharpe={:.3f}  Rounded Sharpe={:.3f}".format(
            res.sharpe,
            raw_validation["summary_row"]["sharpe"],
            rounded_validation["summary_row"]["sharpe"],
        )
    )
    print(
        "Counts: search={} raw_reapply={} rounded={}".format(
            res.trade_count,
            raw_validation["summary_row"]["trade_count"],
            rounded_validation["summary_row"]["trade_count"],
        )
    )
    print(
        "Capital: search={:.2f} raw_reapply={:.2f} rounded={:.2f}".format(
            res.capital_used,
            raw_validation["summary_row"]["capital_used"],
            rounded_validation["summary_row"]["capital_used"],
        )
    )
    print("")

bayesian_candidates_df = pd.DataFrame(bayesian_search_records)
bayesian_candidates_ranked_df, bayesian_candidates_csv_path, bayesian_best_json_path = export_search_artifacts(
    "bayesian_greedy_threshold_search",
    bayesian_candidates_df,
)

print(f"bayesian candidates exported to: {bayesian_candidates_csv_path}")
if bayesian_best_json_path is not None:
    print(f"bayesian best rule exported to: {bayesian_best_json_path}")

bayesian_candidates_ranked_df.head(2)


# 1.5 Load / Apply Chosen Thresholds, Export Dashboards, and Build Review Summary


In [ ]:
selected_search_frames = {
    "greedy_best": greedy_candidates_ranked_df,
    "bayesian_best": bayesian_candidates_ranked_df,
}

selected_rule_rows = {}
summary_rows = []
exported_dashboard_files = []
review_rule_payloads = {}

for rule_name, ranked_df in selected_search_frames.items():
    if ranked_df.empty:
        continue

    top_row = ranked_df.iloc[0].to_dict()
    selected_rule_rows[rule_name] = top_row
    method_name = top_row["method_name"]
    candidate_name = top_row["candidate_name"]
    crit_list = normalize_crit_list(top_row["crit_list_json"])

    partition_dashboard_frames = []
    partition_curve_frames = []
    bydate_map = {}

    for partition_name, partition_df in loaded.items():
        evaluation = summarize_partition_rule(
            partition_df,
            crit_list,
            partition_name,
            method_name,
            candidate_name,
        )
        summary_row = dict(evaluation["summary_row"])
        summary_row["rule_name"] = rule_name
        summary_rows.append(summary_row)

        if not evaluation["dashboard_df"].empty:
            dashboard_export_df = evaluation["dashboard_df"].copy()
            dashboard_export_df["partition_name"] = partition_name
            dashboard_export_df["rule_name"] = rule_name
            dashboard_export_df["candidate_name"] = candidate_name
            partition_dashboard_frames.append(dashboard_export_df)

            if save_partition_dashboards_debug:
                dashboard_filename = f"dashboard_{rule_name}_{PARTITION_OUTPUT_LABELS[partition_name]}.csv"
                exported_dashboard_files.append(save_dataframe_csv(dashboard_export_df, dashboard_filename))

        if not evaluation["dashboard_bydate_df"].empty:
            bydate_export_df = evaluation["dashboard_bydate_df"].copy()
            bydate_export_df["partition_name"] = partition_name
            bydate_export_df["rule_name"] = rule_name
            bydate_export_df["candidate_name"] = candidate_name
            partition_curve_frames.append(bydate_export_df)
            bydate_map[partition_name] = evaluation["dashboard_bydate_df"].copy()

            if save_partition_dashboards_debug and save_equity_curve:
                bydate_filename = f"dashboard_{rule_name}_{PARTITION_OUTPUT_LABELS[partition_name]}_equity_curve.csv"
                exported_dashboard_files.append(save_dataframe_csv(bydate_export_df, bydate_filename))

    if partition_dashboard_frames:
        combined_dashboard_df = pd.concat(partition_dashboard_frames, ignore_index=True)
        dashboard_path = save_dataframe_csv(combined_dashboard_df, f"dashboard_{rule_name}_all_partitions.csv")
        exported_dashboard_files.append(dashboard_path)
    else:
        combined_dashboard_df = pd.DataFrame()

    if partition_curve_frames:
        combined_curve_df = pd.concat(partition_curve_frames, ignore_index=True)
        if save_equity_curve:
            curve_path = save_dataframe_csv(combined_curve_df, f"dashboard_{rule_name}_equity_curves.csv")
            exported_dashboard_files.append(curve_path)
    else:
        combined_curve_df = pd.DataFrame()

    review_rule_payloads[rule_name] = {
        "selected_row": top_row,
        "combined_dashboard_df": combined_dashboard_df,
        "combined_curve_df": combined_curve_df,
        "bydate_map": bydate_map,
    }

threshold_validation_summary = pd.DataFrame(summary_rows)
group_cols = ["method_name", "candidate_name", "rule_name"]
threshold_validation_summary["cross_partition_positive_sharpe"] = threshold_validation_summary.groupby(group_cols)["passes_positive_sharpe"].transform("all")
threshold_validation_summary["cross_partition_min_trades"] = threshold_validation_summary.groupby(group_cols)["passes_min_trades"].transform("all")
threshold_validation_summary["cross_partition_monotonicity"] = threshold_validation_summary.groupby(group_cols)["passes_monotonicity"].transform("all")
threshold_validation_summary["cross_partition_validation_pass"] = threshold_validation_summary.groupby(group_cols)["validation_pass"].transform("all")

threshold_validation_summary_path = save_dataframe_csv(
    threshold_validation_summary,
    "threshold_validation_summary.csv",
)

print("selected rules:")
display(
    pd.DataFrame(
        [
            {
                "rule_name": rule_name,
                "method_name": row["method_name"],
                "candidate_name": row["candidate_name"],
                "crit_list_json": row["crit_list_json"],
                "rounded_sharpe_ins_80": row["rounded_sharpe_ins_80"],
                "rounded_trade_count_ins_80": row["rounded_trade_count_ins_80"],
                "rounded_capital_used_ins_80": row["rounded_capital_used_ins_80"],
            }
            for rule_name, row in selected_rule_rows.items()
        ]
    )
)
print("")
print(f"summary exported to: {threshold_validation_summary_path}")
print("dashboard exports:")
for dashboard_path in exported_dashboard_files:
    print(dashboard_path)

threshold_validation_summary.sort_values(["method_name", "partition_name"]).reset_index(drop=True)


# 1.6 Notes and Preserved Interpretation

The notebook still treats the following research notes as binding review criteria:

- `# Notes: optimization thresholds not sucessful as only the ins_80 has positive Sharpe`
- `# Notes: optimization thresholds not sucessful as equity curves need to be monotonically increasing`

The exported candidate tables and `threshold_validation_summary.csv` now surface these constraints explicitly through pass/fail review columns, instead of leaving them only as notebook-side interpretation.


In [ ]:
# Notes: optimization thresholds not successful as only the ins_80 has positive Sharpe
# Notes: optimization thresholds not successful as equity curves need to be monotonically increasing
#
# Manual reference rules preserved from the original notebook for research continuity.
# They are not treated as automatic winners in this pass.

manual_reference_rules = {
    "manual_reference_1": [
        {
            "ml_proba_1": (0.449, ">="),
            "vol_5d": (0.1043, "<="),
            "spy_rvol": (1.0461, "<="),
            "fear_greed": (46.0, ">="),
        }
    ],
    "manual_reference_2": [
        {
            "ml_proba_1": (0.449, ">="),
            "ret_px_1m_ago": (-0.0196, "<="),
            "ret_px_prev3": (-0.0011, "<="),
        }
    ],
}

manual_reference_rules

for rule_name, payload in review_rule_payloads.items():
    print(f"# Grid Equity Curves: {rule_name}")
    fig, axs = plt.subplots(1, 3, figsize=(20, 5))
    datasets = [payload["bydate_map"].get(partition_name, pd.DataFrame()) for partition_name in PARTITION_NAMES]
    titles = [
        f"{rule_name}: 80% Optmz",
        f"{rule_name}: 20% Optmz",
        f"{rule_name}: OOS Optmz",
    ]

    for ax, data, title in zip(axs, datasets, titles):
        if data is None or data.empty:
            ax.set_title(f"{title} (no data)")
            ax.set_axis_off()
            continue
        line_chart_grid(ax, data, "normed_date", "cum_pl_g", "cum_pl_n", "$ eqt", title)

    plt.tight_layout()
    plt.show()
